# Create runlist CSVs for the multi-SRA-per-pod weebill workflow

This notebook builds the CSV files that feed `sra_multi_workflow_template.yaml`.

Unlike the chunked workflow (`prod_workflow_template4.yaml` + `batch_creation 202602.ipynb`),
here each **pod processes several whole SRA runs** in a loop (like the Logan workflow),
with **no read chunking**. Per accession the pod streams `sracat-rs` reads straight into
`weebill sketch --merge` via FIFOs (no fastq temp files). To stop a node being swamped when
many download-heavy pods land on it at once, we pack accessions into pods with a size budget
and emit an `ephemeral_storage_mb` request per pod. The k8s scheduler bin-packs pods by that
request, so it will not overcommit a node's local disk. Accessions are stratified by run
size before packing so a pod only ever holds similar-sized runs and its disk footprint stays
consistent (see below).

Output CSV columns (one row = one pod):

| column | fed into template parameter | meaning |
| --- | --- | --- |
| `pod_accessions` | `SRA_accession_num` | space-separated list of accessions the pod loops over |
| `batch_name` | `batch_name` | S3 output prefix |
| `ephemeral_storage_mb` | `ephemeral_storage_mb` | requested local disk (MiB) = scheduler size constraint |
| `pod_id` | (naming only) | 0-based pod index within the batch |
| `n_accessions` | (info) | number of accessions in the pod |
| `total_mbytes` | (info) | summed SRA file size of the pod's accessions |
| `max_mbytes` | (info) | largest single run in the pod (drives `ephemeral_storage_mb`) |
| `size_band` | (info) | run-size band the pod's accessions were grouped into |

In [1]:
import polars as pl
import os

In [2]:
def show_all(df, width=200, max_col_width=True):
    '''Print an entire polars dataframe in the console or notebook output.'''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        if max_col_width or len(df.columns) == 1:
            cfg.set_fmt_str_lengths(width)
        print(df)

## Gather the list of metagenomes we need to process

Same inputs as `batch_creation 202602.ipynb`: the full SRA metadata dump, minus everything
already processed in previous runs.

In [3]:
# Read in list of all the metagenomes we need to process
columns = ['acc','releasedate','organism','mbytes','avespotlen','bases','librarylayout']
all_metags = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20260303.some_columns.csv.gz', has_header=False)
all_metags.columns = columns
all_metags.head(), all_metags.shape

(shape: (5, 7)
 ┌─────────────┬────────────────┬────────────────┬────────┬────────────┬────────────┬───────────────┐
 │ acc         ┆ releasedate    ┆ organism       ┆ mbytes ┆ avespotlen ┆ bases      ┆ librarylayout │
 │ ---         ┆ ---            ┆ ---            ┆ ---    ┆ ---        ┆ ---        ┆ ---           │
 │ str         ┆ str            ┆ str            ┆ i64    ┆ i64        ┆ i64        ┆ str           │
 ╞═════════════╪════════════════╪════════════════╪════════╪════════════╪════════════╪═══════════════╡
 │ ERR15880073 ┆ 2025-11-19T00: ┆ wastewater     ┆ 810    ┆ 216        ┆ 2651825214 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR16203935 ┆ 2021-10-05T00: ┆ tick           ┆ 769    ┆ 302        ┆ 1825027978 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR34723171 ┆ 2025-12-01T00: ┆ human          ┆ 2440   ┆ 302    

In [4]:
# Read in list of metagenomes already processed
# prev = pl.read_csv('~/m/msingle/mess/193_202502_sra_scrape/find_gz20250402.accessions.uniq.txt', has_header=False)
# prev.columns = ['acc']
# prev.head(), prev.shape

In [5]:
# Subtract the two to get the list of metagenomes we still need to process
# to_process = all_metags.filter(~pl.col('acc').is_in(prev['acc']))
to_process = all_metags
to_process.head(), to_process.shape

(shape: (5, 7)
 ┌─────────────┬────────────────┬────────────────┬────────┬────────────┬────────────┬───────────────┐
 │ acc         ┆ releasedate    ┆ organism       ┆ mbytes ┆ avespotlen ┆ bases      ┆ librarylayout │
 │ ---         ┆ ---            ┆ ---            ┆ ---    ┆ ---        ┆ ---        ┆ ---           │
 │ str         ┆ str            ┆ str            ┆ i64    ┆ i64        ┆ i64        ┆ str           │
 ╞═════════════╪════════════════╪════════════════╪════════╪════════════╪════════════╪═══════════════╡
 │ ERR15880073 ┆ 2025-11-19T00: ┆ wastewater     ┆ 810    ┆ 216        ┆ 2651825214 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR16203935 ┆ 2021-10-05T00: ┆ tick           ┆ 769    ┆ 302        ┆ 1825027978 ┆ PAIRED        │
 │             ┆ 00:00+00:00    ┆ metagenome     ┆        ┆            ┆            ┆               │
 │ SRR34723171 ┆ 2025-12-01T00: ┆ human          ┆ 2440   ┆ 302    

## Pack accessions into pods, stratified by run size

`mbytes` is the SRA download size. A pod processes its accessions one at a time and
requests `ephemeral_storage_mb` sized for the *largest* run it holds. If a pod mixed a
30 GB run with a handful of 1 GB runs, it would sit on 40+ GB of reserved disk while
churning through the small ones — the node's disk goes up and down and is badly
under-used for most of the pod's life. To keep each pod's disk footprint roughly
constant over its whole run, we group runs of similar size together. We:

1. Drop runs bigger than `MAX_SINGLE_RUN_MBYTES` (they belong in the chunked workflow,
   not this whole-run one, and would blow a single pod's disk).
2. **Stratify** the rest into size bands by `mbytes` (`SIZE_BAND_EDGES_MBYTES`): by
   default 0-10, 10-20, 20-30, 30-50 GB, and >50 GB.
3. Within each band, first-fit-decreasing bin-pack into pods, capping both the summed
   `mbytes` (`MAX_POD_MBYTES`) and the count (`MAX_ACCS_PER_POD`). A pod therefore only
   ever holds runs from one band, so every accession in it is close in size — the pod's
   disk usage stays consistent. Large runs still end up ~alone; many small runs share a
   pod.
4. Set `ephemeral_storage_mb = max_mbytes * STORAGE_SAFETY_FACTOR + STORAGE_HEADROOM_MB`,
   where `max_mbytes` is the largest single run in the pod. Because the pod is
   size-banded, `max_mbytes` is representative of every run it holds, not an outlier.
   Accessions are processed one at a time and sracat-rs streams reads straight into
   `weebill sketch` via FIFOs (no fastq temp files), so at most one `.sra` file is ever on
   disk; the small safety factor only covers filesystem overhead and the tiny `.sylspc`
   sketch outputs. This value is what the k8s scheduler uses to avoid overloading a node.</cell>

In [6]:
# --- Packing / sizing parameters -------------------------------------------------
MAX_SINGLE_RUN_MBYTES = 100_000   # skip runs larger than this (~100 GB); use the chunked workflow instead
MAX_POD_MBYTES        = 30_000    # size budget per pod (summed download size)
MAX_ACCS_PER_POD      = 50        # cap accessions per pod so a pod is not too long-running
STORAGE_SAFETY_FACTOR = 1.3       # local disk vs raw .sra size. sracat-rs streams into weebill (no
                                  # fastq temp files), so ~1x .sra is enough; 2x is a safe margin.
STORAGE_HEADROOM_MB   = 5000      # fixed headroom (working files, .sylspc sketch outputs)

# Upper edges (in mbytes) of the run-size bands used to stratify runs before packing, so a
# pod only ever holds runs of a similar size and its disk footprint stays consistent over
# its whole run. These are 10/20/30/50 GB, giving bands 0-10, 10-20, 20-30, 30-50 GB, and a
# final open-ended >50 GB band (up to MAX_SINGLE_RUN_MBYTES).
SIZE_BAND_EDGES_MBYTES = [5_000, 10_000, 20_000, 30_000, 50_000]


def _size_band_labels(edges):
    '''Human-readable labels for the bands defined by `edges`, e.g. "10-20GB".'''
    labels, prev = [], 0
    for e in edges:
        labels.append(f"{prev // 1000}-{e // 1000}GB")
        prev = e
    labels.append(f"{prev // 1000}GB+")
    return labels


def _size_band_index(mbytes, edges):
    '''Index of the band `mbytes` falls in: band i is [edges[i-1], edges[i]) (lower inclusive).'''
    for i, e in enumerate(edges):
        if mbytes < e:
            return i
    return len(edges)


def pack_into_pods(df, batch_name,
                   max_pod_mbytes=MAX_POD_MBYTES,
                   max_accs_per_pod=MAX_ACCS_PER_POD,
                   max_single_run_mbytes=MAX_SINGLE_RUN_MBYTES,
                   size_band_edges=SIZE_BAND_EDGES_MBYTES,
                   storage_safety_factor=STORAGE_SAFETY_FACTOR,
                   storage_headroom_mb=STORAGE_HEADROOM_MB):
    '''Stratify accessions into size bands, then first-fit-decreasing bin-pack each band.

    df must have columns 'acc' and 'mbytes'. Runs are grouped into the bands defined by
    `size_band_edges` (mbytes) so a pod only ever contains runs of a similar size, keeping
    its disk footprint consistent over its whole run. Returns one row per pod. The caps
    default to the globals above but can be overridden per call (e.g. a small
    max_accs_per_pod to force a test batch across several pods).
    '''
    runs = (
        df.select('acc', 'mbytes')
          .filter(pl.col('mbytes') <= max_single_run_mbytes)
          .sort('mbytes', descending=True)
    )
    print(f"{runs.height} runs to pack after dropping those > {max_single_run_mbytes} mbytes "
          f"({df.height - runs.height} dropped)")

    labels = _size_band_labels(size_band_edges)

    bins = []  # each bin: dict(accs=[...], total=int, max=int, band=int)
    for acc, mbytes in runs.iter_rows():
        band = _size_band_index(mbytes, size_band_edges)
        placed = False
        for b in bins:
            # only pack into a bin from the same size band, so a pod's runs stay similar-sized
            if (b['band'] == band
                    and b['total'] + mbytes <= max_pod_mbytes
                    and len(b['accs']) < max_accs_per_pod):
                b['accs'].append(acc)
                b['total'] += mbytes
                b['max'] = max(b['max'], mbytes)
                placed = True
                break
        if not placed:
            bins.append({'accs': [acc], 'total': mbytes, 'max': mbytes, 'band': band})

    pods = pl.DataFrame({
        'pod_id': list(range(len(bins))),
        'pod_accessions': [' '.join(b['accs']) for b in bins],
        'n_accessions': [len(b['accs']) for b in bins],
        'total_mbytes': [b['total'] for b in bins],
        'max_mbytes': [b['max'] for b in bins],
        'size_band': [labels[b['band']] for b in bins],
    }).with_columns(
        pl.lit(batch_name).alias('batch_name'),
        # Accessions in a pod are processed one at a time (streamed via FIFOs, no fastq
        # temp files) and the pod is size-banded, so peak local disk use is just the
        # largest single run in the pod, which is representative of all of them.
        (pl.col('max_mbytes') * storage_safety_factor + storage_headroom_mb)
            .cast(pl.Int64).alias('ephemeral_storage_mb'),
    ).select(
        'pod_accessions', 'batch_name', 'ephemeral_storage_mb',
        'pod_id', 'n_accessions', 'total_mbytes', 'max_mbytes', 'size_band',
    )
    return pods

### Distribution of run size

Sanity-check the spread of `mbytes` per run and see where the stratification cut points
fall. The x-axis is linear in GB, capped at 50 GB: the long thin tail of very large runs
is collapsed into a single `>50` overflow column (which is exactly the open-ended `50GB+`
band). The dashed lines mark the lower `SIZE_BAND_EDGES_MBYTES` band edges used by
`pack_into_pods`.

In [7]:
from plotnine import (ggplot, aes, geom_histogram, geom_vline, scale_x_continuous,
                      labs, theme_minimal, theme, element_blank, element_text)

# Distribution of per-run download size, linear axis in GB. The size distribution has a
# long thin tail (a few very large runs), so rather than a log axis we cap the x-axis at
# CAP_GB and collapse everything above it into a single ">50" overflow column -- which is
# exactly the open-ended 50GB+ pod size band.
CAP_GB      = 50   # x-axis cap; runs larger than this go in the ">50" overflow column
BINWIDTH_GB = 2

size_df = to_process.filter(pl.col('mbytes') > 0).select(
    (pl.col('mbytes') / 1000).alias('gb')
).to_pandas()
# clamp the tail into a single overflow bin sitting just past the cap
size_df['gb_clamped'] = size_df['gb'].clip(upper=CAP_GB + BINWIDTH_GB / 2)

dist_plot = (
    ggplot(size_df, aes('gb_clamped'))
    + geom_histogram(binwidth=BINWIDTH_GB, boundary=0, fill='#4C78A8')
    # band edges below the cap (10 / 20 / 30 GB); the 50 GB edge is the overflow boundary
    + geom_vline(xintercept=[e / 1000 for e in SIZE_BAND_EDGES_MBYTES if e / 1000 < CAP_GB],
                 color='#C44E52', linetype='dashed', size=0.4)
    + scale_x_continuous(
        breaks=[0, 10, 20, 30, 40, CAP_GB + BINWIDTH_GB / 2],
        labels=['0', '10', '20', '30', '40', '>50'])
    + labs(
        x='download size per run (GB)',
        y='number of runs',
        title='Distribution of SRA run size across metagenomes',
        caption='dashed lines: pod size-band edges (10 / 20 / 30 GB); '
                'rightmost column aggregates all runs > 50 GB (the 50GB+ band)')
    + theme_minimal()
    + theme(figure_size=(9, 5), panel_grid_minor=element_blank(),
            plot_title=element_text(weight='bold'))
)
dist_plot

ModuleNotFoundError: No module named 'plotnine'

In [8]:
# How many runs fall in each size band (across all runs eligible for this workflow).
_band_labels = _size_band_labels(SIZE_BAND_EDGES_MBYTES)


def _band_expr(edges, labels):
    '''Vectorised polars expression mapping `mbytes` to its size-band label.'''
    expr = pl.when(pl.col('mbytes') < edges[0]).then(pl.lit(labels[0]))
    for i in range(1, len(edges)):
        expr = expr.when(pl.col('mbytes') < edges[i]).then(pl.lit(labels[i]))
    return expr.otherwise(pl.lit(labels[len(edges)]))


eligible = to_process.filter(pl.col('mbytes') <= MAX_SINGLE_RUN_MBYTES)
n_dropped = to_process.height - eligible.height

band_counts = (
    eligible
    .with_columns(_band_expr(SIZE_BAND_EDGES_MBYTES, _band_labels)
                  .cast(pl.Enum(_band_labels)).alias('size_band'))
    .group_by('size_band')
    .agg(pl.len().alias('n_runs'))
    .sort('size_band')
    .with_columns((pl.col('n_runs') / pl.col('n_runs').sum() * 100).round(1).alias('pct'))
)
print(f"{eligible.height} runs across the bands; "
      f"{n_dropped} dropped as > {MAX_SINGLE_RUN_MBYTES} mbytes")
show_all(band_counts)

1083375 runs across the bands; 3 dropped as > 100000 mbytes
shape: (6, 3)
┌───────────┬────────┬──────┐
│ size_band ┆ n_runs ┆ pct  │
│ ---       ┆ ---    ┆ ---  │
│ enum      ┆ u32    ┆ f64  │
╞═══════════╪════════╪══════╡
│ 0-5GB     ┆ 976235 ┆ 90.1 │
│ 5-10GB    ┆ 77098  ┆ 7.1  │
│ 10-20GB   ┆ 23066  ┆ 2.1  │
│ 20-30GB   ┆ 4544   ┆ 0.4  │
│ 30-50GB   ┆ 2073   ┆ 0.2  │
│ 50GB+     ┆ 359    ┆ 0.0  │
└───────────┴────────┴──────┘


In [9]:
def generate_and_write_batch(batch_name, to_process, blacklist_accs, num_acc_to_select,
                             making_batch=True, outdir='runlists_multi_sra', seed=42, **pack_kwargs):
    '''Choose accessions, pack them into pods, and write the pod CSV.

    Extra keyword args (e.g. max_accs_per_pod) are forwarded to pack_into_pods.
    '''
    os.makedirs(outdir, exist_ok=True)
    csv_path = os.path.join(outdir, f'{batch_name}.csv')

    if making_batch:
        possible = to_process.filter(~pl.col('acc').is_in(blacklist_accs))
        print(f"{possible.height} accessions available after blacklist of {len(blacklist_accs)}")
        chosen = possible.sample(min(num_acc_to_select, possible.height), seed=seed)
        pods = pack_into_pods(chosen, batch_name, **pack_kwargs)
        # randomise pod order so big and small pods are interleaved on submission
        pods = pods.sample(fraction=1, seed=seed, shuffle=True)
        pods.write_csv(csv_path)
        print(f"Wrote {pods.height} pods covering {pods['n_accessions'].sum()} accessions to {csv_path}")

    return pl.read_csv(csv_path)

### batch1 - a real batch of many small/medium runs

Adjust `num_acc_to_select` to control the batch size.

In [10]:
# batch1 = generate_and_write_batch(
#     batch_name='multi_batch1',
#     to_process=to_process,
#     blacklist_accs=[],
#     num_acc_to_select=10,
#     making_batch=True,
#     max_accs_per_pod=4,
#     max_pod_mbytes=10_000,  # 10 GB per pod
# )
# print(batch1.shape)
# print(batch1.select('n_accessions', 'total_mbytes', 'max_mbytes', 'ephemeral_storage_mb').describe())
# batch1#.head()

# Read in batch1 from the CSV written by the above code, rather than re-picking
batch1 = pl.read_csv('runlists_multi_sra/multi_batch1.csv')
batch1

pod_accessions,batch_name,ephemeral_storage_mb,pod_id,n_accessions,total_mbytes,max_mbytes,size_band
str,str,i64,i64,i64,i64,i64,str
"""SRR17205146 SRR11073009 SRR313…","""multi_batch1""",8875,2,4,6056,2981,"""0-5GB"""
"""ERR10688610 SRR35931871 SRR239…","""multi_batch1""",10804,1,4,9809,4465,"""0-5GB"""
"""DRR806302""","""multi_batch1""",12478,0,1,5753,5753,"""5-10GB"""
"""ERR13867884""","""multi_batch1""",5448,3,1,345,345,"""0-5GB"""


## Submit

Each CSV row becomes one Argo workflow. The `.sylref` reference DB is the same for
the whole batch and is passed as one URI (node-cached, not per pod). For a quick
one-off, substitute the columns into `sra_multi_workflow_template.yaml` and `argo submit`:

```
sed -e "s|{{workflow.parameters.SRA_accession_num}}|$pod_accessions|" \
    -e "s|{{workflow.parameters.batch_name}}|$batch_name|" \
    -e "s|{{workflow.parameters.ephemeral_storage_mb}}|$ephemeral_storage_mb|" \
    -e "s|{{workflow.parameters.reference_s3_uri}}|s3://woodcrob-sandpiper-us-east-1/db/r232.top50000.100.with_genomes.sylref|" \
    sra_weebill_workflow_template.yaml | argo submit -n argo -
```

For throttled, resumable submission use `slow_argo_submission_weebill.py`, which reads the CSV
and replaces the same placeholders per pod (with `--blacklist` to drop already-finished
accessions):

```
pixi run ./slow_argo_submission_weebill.py \
    --input-runlist-csv runlists_multi_sra/multi_batch3.csv \
    --workflow-template sra_weebill_workflow_template.yaml \
    --reference-s3-uri s3://woodcrob-sandpiper-us-east-1/db/r232.top50000.100.with_genomes.sylref \
    --batch-size 200 --min-running-pending-file min_job_count
```

In [11]:
# Another 100 candidates
# batch2 = generate_and_write_batch(
#     batch_name='multi_batch2',
#     to_process=to_process,
#     blacklist_accs=[],
#     num_acc_to_select=100,
#     making_batch=True,
#     max_accs_per_pod=10,
#     # max_pod_mbytes=10_000,  # 10 GB per pod
# )
# print(batch2.shape)
# print(batch2.select('n_accessions', 'total_mbytes', 'max_mbytes', 'ephemeral_storage_mb').describe())
# batch2.head()

# Read in batch1 from the CSV written by the above code, rather than re-picking
batch2 = pl.read_csv('runlists_multi_sra/multi_batch2.csv')
print(batch2.shape)
batch2.head()

(16, 8)


pod_accessions,batch_name,ephemeral_storage_mb,pod_id,n_accessions,total_mbytes,max_mbytes,size_band
str,str,i64,i64,i64,i64,i64,str
"""SRR35029147 SRR17722506 SRR354…","""multi_batch2""",5620,14,10,3746,477,"""0-5GB"""
"""ERR13243068 SRR19422192 ERR719…","""multi_batch2""",5828,13,10,5532,637,"""0-5GB"""
"""SRR23978806""","""multi_batch2""",18072,4,1,10056,10056,"""10-20GB"""
"""SRR33600036 SRR37036535""","""multi_batch2""",12189,6,2,10688,5530,"""5-10GB"""
"""ERR3211936 ERR5925364 SRR30731…","""multi_batch2""",7769,9,10,17871,2130,"""0-5GB"""


In [12]:
# Another 100 candidates
# batch3 = generate_and_write_batch(
#     batch_name='multi_batch3',
#     to_process=to_process,
#     blacklist_accs=[],
#     num_acc_to_select=100,
#     making_batch=True,
#     max_accs_per_pod=10,
#     # max_pod_mbytes=10_000,  # 10 GB per pod
# )
# print(batch3.shape)
# print(batch3.select('n_accessions', 'total_mbytes', 'max_mbytes', 'ephemeral_storage_mb').describe())
# batch3.head()

# Read in batch1 from the CSV written by the above code, rather than re-picking
batch3 = pl.read_csv('runlists_multi_sra/multi_batch3.csv')


In [13]:
# Another 1000 candidates
processed_accs = batch1['pod_accessions'].str.split(' ').explode().unique().to_list() + \
                 batch2['pod_accessions'].str.split(' ').explode().unique().to_list() + \
                 batch3['pod_accessions'].str.split(' ').explode().unique().to_list()
batch4 = generate_and_write_batch(
    batch_name='multi_batch4',
    to_process=to_process,
    blacklist_accs=processed_accs,
    num_acc_to_select=1000,
    making_batch=True,
    # max_accs_per_pod=10,
    # max_pod_mbytes=10_000,  # 10 GB per pod
)
print(batch4.shape)
print(batch4.select('n_accessions', 'total_mbytes', 'max_mbytes', 'ephemeral_storage_mb').describe())
batch4.head()

1083268 accessions available after blacklist of 210
1000 runs to pack after dropping those > 100000 mbytes (0 dropped)
Wrote 89 pods covering 1000 accessions to runlists_multi_sra/multi_batch4.csv
(89, 8)
shape: (9, 5)
┌────────────┬──────────────┬──────────────┬─────────────┬──────────────────────┐
│ statistic  ┆ n_accessions ┆ total_mbytes ┆ max_mbytes  ┆ ephemeral_storage_mb │
│ ---        ┆ ---          ┆ ---          ┆ ---         ┆ ---                  │
│ str        ┆ f64          ┆ f64          ┆ f64         ┆ f64                  │
╞════════════╪══════════════╪══════════════╪═════════════╪══════════════════════╡
│ count      ┆ 89.0         ┆ 89.0         ┆ 89.0        ┆ 89.0                 │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0                  │
│ mean       ┆ 11.235955    ┆ 27746.876404 ┆ 7677.179775 ┆ 14979.898876         │
│ std        ┆ 12.324239    ┆ 5559.729355  ┆ 8539.112868 ┆ 11100.874386         │
│ min        ┆ 1.0          ┆ 1073.0       

pod_accessions,batch_name,ephemeral_storage_mb,pod_id,n_accessions,total_mbytes,max_mbytes,size_band
str,str,i64,i64,i64,i64,i64,str
"""SRR29284213 SRR29257317 SRR325…","""multi_batch4""",11802,37,5,25968,5233,"""5-10GB"""
"""SRR7124683 SRR28210365 SRR3067…","""multi_batch4""",9862,50,9,30000,3740,"""0-5GB"""
"""SRR22387980""","""multi_batch4""",18149,21,1,10115,10115,"""10-20GB"""
"""ERR11733733 SRR21663007""","""multi_batch4""",27392,10,2,29971,17225,"""10-20GB"""
"""ERR3870193 SRR37225935 SRR6075…","""multi_batch4""",11230,40,7,29999,4793,"""0-5GB"""


In [14]:
# Another 5000 candidates, excluding every accession already in batches 1-4
processed_accs_5 = batch1['pod_accessions'].str.split(' ').explode().unique().to_list() + \
                   batch2['pod_accessions'].str.split(' ').explode().unique().to_list() + \
                   batch3['pod_accessions'].str.split(' ').explode().unique().to_list() + \
                   batch4['pod_accessions'].str.split(' ').explode().unique().to_list()
batch5 = generate_and_write_batch(
    batch_name='multi_batch5',
    to_process=to_process,
    blacklist_accs=processed_accs_5,
    num_acc_to_select=5000,
    making_batch=True,
)
print(batch5.shape)
print(batch5.select('n_accessions', 'total_mbytes', 'max_mbytes', 'ephemeral_storage_mb').describe())
batch5.head()

1082268 accessions available after blacklist of 1210
5000 runs to pack after dropping those > 100000 mbytes (0 dropped)
Wrote 436 pods covering 5000 accessions to runlists_multi_sra/multi_batch5.csv
(436, 8)
shape: (9, 5)
┌────────────┬──────────────┬──────────────┬────────────┬──────────────────────┐
│ statistic  ┆ n_accessions ┆ total_mbytes ┆ max_mbytes ┆ ephemeral_storage_mb │
│ ---        ┆ ---          ┆ ---          ┆ ---        ┆ ---                  │
│ str        ┆ f64          ┆ f64          ┆ f64        ┆ f64                  │
╞════════════╪══════════════╪══════════════╪════════════╪══════════════════════╡
│ count      ┆ 436.0        ┆ 436.0        ┆ 436.0      ┆ 436.0                │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0        ┆ 0.0                  │
│ mean       ┆ 11.46789     ┆ 28231.002294 ┆ 6946.62156 ┆ 14030.142202         │
│ std        ┆ 12.254068    ┆ 4554.159736  ┆ 7505.44001 ┆ 9757.077515          │
│ min        ┆ 1.0          ┆ 1270.0       ┆ 46.0

pod_accessions,batch_name,ephemeral_storage_mb,pod_id,n_accessions,total_mbytes,max_mbytes,size_band
str,str,i64,i64,i64,i64,i64,str
"""SRR24155128 SRR31377730 ERR160…","""multi_batch5""",7837,327,14,30000,2183,"""0-5GB"""
"""ERR11733649 ERR12652045""","""multi_batch5""",20997,74,2,23773,12306,"""10-20GB"""
"""SRR28401880 SRR22388235 SRR130…","""multi_batch5""",17282,97,3,28215,9448,"""5-10GB"""
"""SRR17241694 SRR33660286 SRR258…","""multi_batch5""",14937,121,4,29987,7644,"""5-10GB"""
"""SRR31386139 SRR18577011 SRR306…","""multi_batch5""",11905,172,5,26489,5312,"""5-10GB"""


In [15]:
# The rest of the backlog: everything NOT already in batches 1-5, split into batches of
# REMAINING_BATCH_ACCS accessions each (multi_batch6, multi_batch7, ...). Unlike batches
# 1-5 (hand-sized random samples), this exhausts the pool: "remaining" = all of
# `to_process`, minus every accession already placed in batches 1-5, minus runs larger
# than MAX_SINGLE_RUN_MBYTES (those belong in the chunked workflow). The pool is shuffled
# once (seed=42) then sliced, so each batch gets a representative mix of run sizes and,
# because the seed is fixed, the partition is reproducible - an accession never lands in
# two batches. Each slice is packed into pods exactly like the other batches.
#
# Set REGENERATE_REMAINING=False to skip repacking ~1M accessions and just read the
# existing CSVs back.
REMAINING_BATCH_ACCS      = 20_000
FIRST_REMAINING_BATCH_NUM = 6
REGENERATE_REMAINING      = True

already_batched = set(
    acc
    for b in (batch1, batch2, batch3, batch4, batch5)
    for acc in b['pod_accessions'].str.split(' ').explode().to_list()
)

remaining = (
    to_process
    .filter(~pl.col('acc').is_in(already_batched))
    .filter(pl.col('mbytes') <= MAX_SINGLE_RUN_MBYTES)
    .sample(fraction=1.0, seed=42, shuffle=True)   # one deterministic shuffle, then slice
)
n_batches = -(-remaining.height // REMAINING_BATCH_ACCS)   # ceil division
print(f"{len(already_batched)} accessions already in batches 1-5; "
      f"{remaining.height} remaining eligible runs -> {n_batches} batches of {REMAINING_BATCH_ACCS}")

remaining_batches = {}
os.makedirs('runlists_multi_sra', exist_ok=True)
for i in range(n_batches):
    batch_name = f'multi_batch{FIRST_REMAINING_BATCH_NUM + i}'
    csv_path = f'runlists_multi_sra/{batch_name}.csv'
    if REGENERATE_REMAINING or not os.path.exists(csv_path):
        chunk = remaining.slice(i * REMAINING_BATCH_ACCS, REMAINING_BATCH_ACCS)
        pods = pack_into_pods(chunk, batch_name)
        pods = pods.sample(fraction=1.0, seed=42, shuffle=True)  # interleave big/small pods
        pods.write_csv(csv_path)
        print(f"  {batch_name}: {pods.height} pods, {pods['n_accessions'].sum()} accessions -> {csv_path}")
    remaining_batches[batch_name] = pl.read_csv(csv_path)

total_pods = sum(b.height for b in remaining_batches.values())
total_accs = sum(int(b['n_accessions'].sum()) for b in remaining_batches.values())
last_num = FIRST_REMAINING_BATCH_NUM + n_batches - 1
print(f"\n{len(remaining_batches)} remaining batches "
      f"(multi_batch{FIRST_REMAINING_BATCH_NUM}..multi_batch{last_num}): "
      f"{total_pods} pods covering {total_accs} accessions")


6110 accessions already in batches 1-5; 1077265 remaining eligible runs -> 54 batches of 20000
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch6: 1685 pods, 20000 accessions -> runlists_multi_sra/multi_batch6.csv
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch7: 1686 pods, 20000 accessions -> runlists_multi_sra/multi_batch7.csv
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch8: 1707 pods, 20000 accessions -> runlists_multi_sra/multi_batch8.csv
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch9: 1684 pods, 20000 accessions -> runlists_multi_sra/multi_batch9.csv
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


...

  multi_batch55: 1680 pods, 20000 accessions -> runlists_multi_sra/multi_batch55.csv
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch56: 1695 pods, 20000 accessions -> runlists_multi_sra/multi_batch56.csv
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch57: 1662 pods, 20000 accessions -> runlists_multi_sra/multi_batch57.csv
20000 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch58: 1676 pods, 20000 accessions -> runlists_multi_sra/multi_batch58.csv
17265 runs to pack after dropping those > 100000 mbytes (0 dropped)


  multi_batch59: 1464 pods, 17265 accessions -> runlists_multi_sra/multi_batch59.csv

54 remaining batches (multi_batch6..multi_batch59): 90810 pods covering 1077265 accessions


## Submit

Each CSV row becomes one Argo workflow. For a quick one-off, substitute the columns into
`sra_multi_workflow_template.yaml` and `argo submit`, e.g. per row:

```
sed -e "s|{{workflow.parameters.SRA_accession_num}}|$pod_accessions|" \
    -e "s|{{workflow.parameters.batch_name}}|$batch_name|" \
    -e "s|{{workflow.parameters.ephemeral_storage_mb}}|$ephemeral_storage_mb|" \
    sra_multi_workflow_template.yaml | argo submit -n argo -
```

For throttled, resumable submission use `slow_argo_submission_multi.py`, which reads the CSV
and replaces the same three placeholders per pod (with `--blacklist` to drop already-finished
accessions):

```
./slow_argo_submission_multi.py \
    --input-runlist-csv runlists_multi_sra/multi_batch1.csv \
    --workflow-template sra_multi_workflow_template.yaml \
    --batch-size 200 --min-running-pending-file min_job_count
```